# Zenith AI — Kaggle Training

Trains the ~100.7M-parameter `configs/zenith_kaggle.yaml` variant of Zenith on a Kaggle GPU (T4 or P100).

This scales up from an earlier 75.5M/12-layer run that validated the architecture and pipeline on Kaggle (val perplexity 19.0 → 8.75 → 6.60 over steps 250/500/750, cleanly improving) before being cut off by the session limit around step ~1500/3000. This config goes deeper (16 layers instead of 12, same width) and caps at 1000 steps so it has a realistic chance of finishing inside one session instead of needing ~14h like the previous 3000-step attempt did.

**Before running:** in the notebook side panel, set **Accelerator = GPU** and **Internet = On** (needed for `git clone` and the TinyStories dataset download).

**Session limit:** Kaggle sessions cap at ~9-12h. Training is checkpointed every 250 steps to `/kaggle/working/checkpoints` and auto-resumes from `latest.pt` if present, so if a run gets cut off, save `/kaggle/working/checkpoints` as a Kaggle Dataset, attach it as input on the next run, copy it back into `checkpoints/`, and re-run this notebook — it will pick up where it left off.

Every shell command below uses an absolute `cd /kaggle/working/Zenith-AI && ...` prefix rather than relying on a notebook-wide working directory. Kaggle's headless (papermill) execution can restart the kernel process mid-run (e.g. after a CUDA/GPU-arch warning), which silently resets `%cd` state — chaining `cd` into each command avoids `ModuleNotFoundError: No module named 'zenith...'` errors from cells accidentally running from `/kaggle/working` instead of the repo root.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

In [ ]:
REPO = "/kaggle/working/Zenith-AI"

# Clone the repo
!rm -rf {REPO}
!git clone --depth 1 https://github.com/ItzAditya43/Zenith-AI.git {REPO}

## Make sure PyTorch supports whatever GPU Kaggle assigned

Kaggle's preinstalled PyTorch build sometimes drops support for older GPU architectures (e.g. Tesla P100, compute capability sm_60) while still *offering* P100 as an accelerator. If that happens, `torch.cuda.is_available()` still returns `True` but every real op silently fails or warns. This cell detects a mismatch and reinstalls a PyTorch/CUDA build that supports the assigned GPU's compute capability.

In [ ]:
import subprocess

def gpu_compute_capability():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip().splitlines()
    return out[0] if out else None

cc = gpu_compute_capability()
print("GPU compute capability:", cc)

# sm_60 (Pascal, e.g. P100) needs an older CUDA 11.8-era PyTorch build; sm_70+ (T4, V100, etc.) works with the stock image.
if cc is not None and float(cc) < 7.0:
    print("Older GPU architecture detected — reinstalling a compatible PyTorch build (cu118)...")
    !pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu118
else:
    print("Stock PyTorch build should support this GPU, skipping reinstall.")

In [ ]:
# Extra deps not in the base Kaggle image
!pip install -q tokenizers datasets pyyaml tqdm

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    # Actually exercise the GPU rather than trusting is_available() alone
    x = torch.randn(4, 4, device="cuda") @ torch.randn(4, 4, device="cuda")
    torch.cuda.synchronize()
    print("CUDA matmul smoke test OK:", x.shape)
else:
    raise RuntimeError("No GPU available — set Accelerator = GPU in the notebook settings panel and rerun.")

## Optional: resume from a previous session

If you attached a Kaggle Dataset containing a prior `checkpoints/` output, copy it in before training starts. Otherwise skip this cell — training starts fresh.

In [ ]:
# import shutil, os
# prev_ckpt_dataset = "/kaggle/input/<your-attached-checkpoint-dataset>"
# os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
# for f in os.listdir(prev_ckpt_dataset):
#     shutil.copy(os.path.join(prev_ckpt_dataset, f), "/kaggle/working/checkpoints/")
# print("Restored checkpoints:", os.listdir("/kaggle/working/checkpoints"))

## Data pipeline

Downloads TinyStories, trains an 8192-vocab BPE tokenizer from scratch, then tokenizes and packs it into fixed-length shards. Skipped automatically on resume if the packed `.npy` files already exist.

In [ ]:
import os
if not os.path.exists(f"{REPO}/data/train.txt"):
    !cd {REPO} && python -m zenith.data.prepare dump --output-dir data

In [ ]:
if not os.path.exists(f"{REPO}/data/tokenizer.json"):
    !cd {REPO} && python -m zenith.tokenizer.train_tokenizer --input data/train.txt --output data/tokenizer.json --vocab-size 8192

In [ ]:
# Full TinyStories corpus this time (Kaggle has the disk + GPU headroom for it).
# Context length is 512 here (vs. 256 in the local laptop config) — must repack even if you ran the smaller config before.
if not os.path.exists(f"{REPO}/data/train_packed.npy"):
    !cd {REPO} && python -m zenith.data.prepare pack --input data/train.txt --tokenizer data/tokenizer.json --output data/train_packed.npy --seq-len 512
if not os.path.exists(f"{REPO}/data/val_packed.npy"):
    !cd {REPO} && python -m zenith.data.prepare pack --input data/val.txt --tokenizer data/tokenizer.json --output data/val_packed.npy --seq-len 512

## Train

~100.7M params (16 layers). Capped at 1000 steps to fit inside one Kaggle commit session, based on the ~3780 tok/s measured on a shared T4 during the previous 75.5M run. Checkpoints land in `/kaggle/working/checkpoints/latest.pt` (auto-resumed if this cell reruns) and are included in the notebook's persistent output automatically.

In [ ]:
!cd {REPO} && python -m zenith.training.train configs/zenith_kaggle.yaml

## Try it

In [ ]:
import sys
sys.path.insert(0, REPO)
os.chdir(REPO)

from zenith.cli import load_model_and_tokenizer
from zenith.inference.generate import generate_text

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tok, cfg = load_model_and_tokenizer("/kaggle/working/checkpoints/latest.pt", "data/tokenizer.json", device)

for prompt in ["Once upon a time", "The little dog was very", "Tom and Lily went to the park"]:
    text = generate_text(model, tok, prompt, max_new_tokens=100, temperature=0.7, top_k=40, device=device)
    print("PROMPT:", prompt)
    print("OUTPUT:", prompt + text)
    print()

## Save checkpoints as a Dataset (for resuming later)

Kaggle keeps `/kaggle/working` as notebook output automatically when you **Save Version**. To resume in a future session: after saving this version, go to "New Dataset" → source it from this notebook's output → attach that dataset as input next time → uncomment the restore cell above.